# Exploring CABIQAE — arXiv:2607.12990
**ArXivist-generated exploratory notebook**

This notebook visualizes the paper's central algorithmic idea: how contrast
decay reshapes the Grover-amplified observation model, and how CABIQAE's
error-vs-query-cost scaling compares to a noise-naive baseline (BIQAE) and
direct circuit sampling (DCS) once that decay is present. All plots use
synthetic/toy circuits so nothing here requires pretrained checkpoints or
hardware access -- swap in `results/comparison_*.json` from `evaluate.py`
to reproduce the paper's actual Figure 6 / Figure 10 style plots at full scale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from quantum_cva.estimation import CABIQAE, BIQAE, DirectCircuitSampling

plt.rcParams["figure.figsize"] = (7, 4.5)


## 1. The contrast-decay model (Eq. 40)

$$p_{obs}(\vartheta,k) = b + c(k)\left(\sin^2(K\vartheta)-b\right), \qquad c(k)=c_0 e^{-K/\tau_c}, \quad K=2k+1$$

As $\tau_c \to \infty$ this recovers the ideal Grover oscillation. The paper's
fitted values are $\tau_c \approx 34$ for the shallow validation circuit
(Table 17) vs. $\tau_c \approx 2.2$ for the much deeper full CVA oracle
(Table 18) -- amplification is far less useful once circuits get that deep.

In [ ]:
theta_true = np.arcsin(np.sqrt(0.36))
k_values = np.arange(0, 40)
K_values = 2 * k_values + 1

fig, ax = plt.subplots()
for tau_c, label, color in [(1e12, "ideal (tau_c -> inf)", "black"),
                              (33.96, "validation circuit (tau_c=34)", "tab:blue"),
                              (2.2, "full CVA oracle (tau_c=2.2)", "tab:red")]:
    est = CABIQAE(c0=1.0, tau_c=tau_c, b=0.5)
    p_obs = est.p_obs(np.full_like(K_values, theta_true, dtype=float), k=0)  # placeholder
    p_obs = np.array([est.p_obs(np.array([theta_true]), k)[0] for k in k_values])
    ax.plot(K_values, p_obs, label=label, color=color, linewidth=2 if tau_c < 1e6 else 1, linestyle="--" if tau_c > 1e6 else "-")

ax.set_xlabel("Amplification depth K = 2k+1")
ax.set_ylabel("Observed success probability")
ax.set_title("Grover-contrast decay (Eq. 40)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 2. Error-vs-query-cost scaling: CABIQAE vs BIQAE vs DCS

Reproduces the qualitative shape of the paper's Figure 6/10: in the ideal
regime, amplified methods (CABIQAE, BIQAE) follow $O(1/N_q)$ while direct
sampling (DCS) follows the slower classical $O(1/\sqrt{N_q})$ rate. Under
contrast decay, the noise-naive BIQAE degrades while CABIQAE keeps improving
over a longer (though still bounded) query range.

In [ ]:
def run_trajectories(estimator_factory, executor_factory, epsilons, n_repeats=5, **kwargs):
    """Run several epsilon targets x repeats, return (N_q, median_rel_error) points."""
    points = []
    for eps in epsilons:
        errs, nqs = [], []
        for rep in range(n_repeats):
            rng = np.random.default_rng(hash((eps, rep)) % (2**31))
            executor = executor_factory(rng)
            estimator = estimator_factory()
            result = estimator.estimate(executor, epsilon=eps, **kwargs)
            errs.append(abs(result.a_hat - a_true) / a_true)
            nqs.append(result.total_queries)
        points.append((np.median(nqs), np.median(errs)))
    return np.array(points)


a_true = 0.36
theta_true_local = np.arcsin(np.sqrt(a_true))
epsilons = [0.08, 0.05, 0.03, 0.02, 0.015, 0.01]

# --- Ideal regime ---
def ideal_executor_factory(rng):
    def executor(k, n_shots):
        K = 2 * k + 1
        p = np.sin(K * theta_true_local) ** 2
        return int(rng.binomial(n_shots, p)), n_shots
    return executor

cabiqae_ideal_pts = run_trajectories(
    lambda: CABIQAE(c0=1.0, tau_c=1e12, b=0.5, rho_min=2.0),
    ideal_executor_factory, epsilons, n_repeats=5, alpha=0.1, n_batch=128, max_stages=50,
)
biqae_ideal_pts = run_trajectories(
    lambda: BIQAE(rho_min=2.0),
    ideal_executor_factory, epsilons, n_repeats=5, alpha=0.1, n_batch=128, max_stages=50,
)

print("CABIQAE (ideal) N_q, error points:")
print(cabiqae_ideal_pts)


In [ ]:
# --- DCS (direct circuit sampling), unamplified baseline ---
dcs = DirectCircuitSampling()
dcs_pts = []
rng_dcs = np.random.default_rng(123)
for n_shots in [200, 500, 1000, 3000, 8000, 20000]:
    result = dcs.estimate(ideal_executor_factory(rng_dcs), n_shots=n_shots)
    err = abs(result.a_hat - a_true) / a_true
    dcs_pts.append((result.total_queries, err))
dcs_pts = np.array(dcs_pts)

fig, ax = plt.subplots()
ax.loglog(cabiqae_ideal_pts[:, 0], cabiqae_ideal_pts[:, 1], "o-", label="CABIQAE (ideal)")
ax.loglog(biqae_ideal_pts[:, 0], biqae_ideal_pts[:, 1], "s-", label="BIQAE (ideal)")
ax.loglog(dcs_pts[:, 0], dcs_pts[:, 1], "^-", label="DCS (unamplified)")

# Reference slopes
n_ref = np.array([cabiqae_ideal_pts[:, 0].min(), cabiqae_ideal_pts[:, 0].max()])
y0 = cabiqae_ideal_pts[0, 1]
ax.loglog(n_ref, y0 * (n_ref[0] / n_ref) ** 1, "k--", alpha=0.5, label="O(1/N)")
ax.loglog(n_ref, y0 * (n_ref[0] / n_ref) ** 0.5, "k:", alpha=0.5, label="O(1/sqrt(N))")

ax.set_xlabel("Query cost N_q")
ax.set_ylabel("Median relative error")
ax.set_title("Ideal regime: amplified O(1/N) vs classical O(1/sqrt(N)) (cf. paper Fig. 6 top)")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()


In [ ]:
# --- Hardware-replay regime (short contrast length) ---
def hw_executor_factory(rng, c0=1.0, tau_c=3.0, b=0.5):
    def executor(k, n_shots):
        K = 2 * k + 1
        q_k = np.sin(K * theta_true_local) ** 2
        c_k = c0 * np.exp(-K / tau_c)
        p_hw = np.clip(b + c_k * (q_k - b), 0, 1)
        return int(rng.binomial(n_shots, p_hw)), n_shots
    return executor

cabiqae_hw_pts = run_trajectories(
    lambda: CABIQAE(c0=1.0, tau_c=3.0, b=0.5, rho_min=2.0),
    hw_executor_factory, epsilons, n_repeats=5, alpha=0.1, n_batch=128, max_stages=50,
)
biqae_hw_pts = run_trajectories(
    lambda: BIQAE(rho_min=2.0),
    hw_executor_factory, epsilons, n_repeats=5, alpha=0.1, n_batch=128, max_stages=50,
)

fig, ax = plt.subplots()
ax.loglog(cabiqae_hw_pts[:, 0], cabiqae_hw_pts[:, 1], "o-", label="CABIQAE (contrast-aware)")
ax.loglog(biqae_hw_pts[:, 0], biqae_hw_pts[:, 1], "s-", label="BIQAE (noise-naive)")
ax.set_xlabel("Query cost N_q")
ax.set_ylabel("Median relative error")
ax.set_title("Hardware-replay regime (tau_c=3): CABIQAE degrades gracefully (cf. paper Fig. 6/10 bottom)")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print("Qualitative takeaway (matches paper Section 4.1/4.4): under contrast")
print("decay, the noise-naive BIQAE's error can plateau or worsen as it keeps")
print("trusting deep low-contrast circuits, while CABIQAE's scheduler avoids")
print("over-investing in uninformative depths.")


## 3. Error-budget decomposition (Section 4.5, Eq. 51-56)

Visualizes how the paper's total end-to-end error splits across finite-grid
discretisation, quantum-encoding training error, and the amplitude-estimation
residual -- using the paper's own reported numbers (Table 7).

In [ ]:
from quantum_cva.evaluation import ErrorBudget

budget = ErrorBudget()
result_noiseless = budget.compute_budget(
    cva_cont_mc=1.091, cva_tab=0.522, cva_sv=0.670, cva_ae=0.670 * (1 + 0.0000227)
)
result_hw = budget.compute_budget(
    cva_cont_mc=1.091, cva_tab=0.522, cva_sv=0.670, cva_ae=0.670 * (1 + 0.0515)
)

labels = ["Finite-grid\n(eps_grid)", "Encoding/training\n(alpha_n * eps_enc)", "AE -- noiseless\n(beta * eps_AE)", "AE -- hardware\n(beta * eps_AE)"]
values = [
    result_noiseless.eps_grid,
    result_noiseless.alpha_n * result_noiseless.eps_enc,
    result_noiseless.beta_theta * result_noiseless.eps_ae,
    result_hw.beta_theta * result_hw.eps_ae,
]

fig, ax = plt.subplots()
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
bars = ax.bar(labels, [v * 100 for v in values], color=colors)
ax.set_ylabel("Contribution to total relative CVA error (%)")
ax.set_title("Error-budget decomposition (paper Table 7)")
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, f"{v:.2%}", ha="center")
plt.tight_layout()
plt.show()

print(f"Total bound (noiseless): {result_noiseless.total_bound:.2%}  (paper: ~65.73%)")
print(f"Total bound (hardware):  {result_hw.total_bound:.2%}  (paper: ~68.88%)")
print()
print("Takeaway (Section 4.5): finite-grid discretisation and encoding")
print("training error dominate the total budget; the amplitude-estimation")
print("residual -- even under hardware noise -- remains a secondary contributor.")


## Next steps

- Swap the toy `a_true=0.36` amplitude for your own trained oracle's
  `oracle.marked_amplitude_statevector(a_theta)` (see `reproduce_arxiv_2607_12990.ipynb`).
- Load `results/comparison_hardware-replay.json` from a full `evaluate.py --num-trajectories 300`
  run to overlay real (not toy) trajectory data on these plots.
- Fit real contrast parameters with `run_hardware_calibration.py` and substitute
  them into cell 2 above.
